# Chạy tất cả mô hình D.O.N.G

So sánh Gaussian Naive Bayes, Categorical Naive Bayes, Linear SVM và RBF SVM trên Incribo và CIC-IDS2017. Các mô hình dùng chung cách chia tập trong từng bộ dữ liệu.

## Chọn dữ liệu

Mặc định chạy cả hai bộ trên toàn bộ bản ghi đã làm sạch của ba lớp. Đổi `DATASET` thành `"incribo"` hoặc `"cicids2017"` để chạy một bộ. `FULL_DATA = False` dùng mẫu CIC-IDS2017 cân bằng.

`TUNE_ROWS = 4000` giới hạn mẫu dò siêu tham số. Đặt `TUNE_ROWS = 0` để chạy CV trên toàn bộ tập train, với thời gian chạy dài hơn.

In [1]:
DATASET = "all"
JOBS = 1
FULL_DATA = True
TUNE_ROWS = 0

## Huấn luyện và đánh giá

Tỷ lệ chia phân tầng là 70% train, 10% validation và 20% test. Chương trình chọn siêu tham số theo Macro-F1 CV rồi fit lại trên toàn bộ train và validation. RBF SVM có thể mất vài phút. Mỗi lượt ghi một thư mục mới trong `results/`.

In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / "run_experiment.py").is_file() and (path / "dong_data.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Hãy mở Jupyter từ thư mục D.O.N.G hoặc một thư mục con của dự án.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from run_experiment import main

arguments = ["--dataset", DATASET, "--jobs", str(JOBS), "--tune-rows", str(TUNE_ROWS)]
if FULL_DATA:
    arguments.append("--full-data")

results_path = main(argv=arguments)


[incribo] Preparing data
[incribo] Tuning gaussian_nb on 27,997 training rows
[incribo] Tuning categorical_nb on 27,997 training rows
[incribo] Tuning linear_svm on 27,997 training rows
[incribo] Tuning rbf_svm on 27,997 training rows
[incribo] Evaluating gaussian_nb
[incribo] gaussian_nb test_macro_f1=0.3176
[incribo] Evaluating categorical_nb
[incribo] categorical_nb test_macro_f1=0.3355
[incribo] Evaluating linear_svm
[incribo] linear_svm test_macro_f1=0.3356
[incribo] Evaluating rbf_svm
[incribo] rbf_svm test_macro_f1=0.3388
[incribo] Evaluating majority
[incribo] majority test_macro_f1=0.1676
[incribo] Evaluating uniform
[incribo] uniform test_macro_f1=0.3297
[incribo] Evaluating shuffled_linear_svm
[incribo] shuffled_linear_svm test_macro_f1=0.3267

[cicids2017] Preparing data
[cicids2017] Tuning gaussian_nb on 91,486 training rows
[cicids2017] Tuning categorical_nb on 91,486 training rows
[cicids2017] Tuning linear_svm on 91,486 training rows
[cicids2017] Tuning rbf_svm on 91,4

## Bảng kết quả trên tập test

Precision, Recall và F1-Score là trung bình đều trên ba lớp. MSE tính trên vector one-hot của nhãn thật và nhãn dự đoán. Với ba lớp, MSE bằng `2/3 × (1 − Accuracy)`. Thời gian fit và dự đoán được tính bằng giây.

In [3]:
import pandas as pd
from IPython.display import display

results = pd.read_csv(results_path)
metrics = {
    "accuracy": "Accuracy",
    "macro_precision": "Precision (macro)",
    "macro_recall": "Recall (macro)",
    "macro_f1": "F1-Score (macro)",
    "mse": "MSE (one-hot)",
}
test_labels = {f"test_{metric}": label for metric, label in metrics.items()}
score_columns = ["dataset", "model", *test_labels, "fit_seconds", "predict_seconds"]
display(results[score_columns].rename(columns=test_labels).round(4))

,dataset,model,Accuracy,Precision (macro),Recall (macro),F1-Score (macro),MSE (one-hot),fit_seconds,predict_seconds
0,incribo,gaussian_nb,0.3410,0.3410,0.3401,0.3176,0.4393,0.0211,0.0026
1,incribo,categorical_nb,0.3375,0.3372,0.3373,0.3355,0.4417,0.0729,0.0147
2,incribo,linear_svm,0.3369,0.3364,0.3367,0.3356,0.4421,0.1953,0.0114
3,incribo,rbf_svm,0.3389,0.3389,0.3388,0.3388,0.4408,61.1829,27.5378
4,incribo,majority,0.3358,0.1119,0.3333,0.1676,0.4428,0.0063,0.0002
5,incribo,uniform,0.3298,0.3297,0.3297,0.3297,0.4468,0.0057,0.0003
6,incribo,shuffled_linear_svm,0.3296,0.3297,0.3293,0.3267,0.4469,0.1773,0.0129
7,cicids2017,gaussian_nb,0.9873,0.9668,0.7419,0.7773,0.0084,0.1375,0.0126
8,cicids2017,categorical_nb,0.9978,0.9896,0.9622,0.9753,0.0015,0.2122,0.0164
9,cicids2017,linear_svm,0.9992,0.9856,0.9974,0.9914,0.0006,7.9883,0.0130


## Kết quả cross-validation

CV phân tầng 5 fold với 3 seed thực hiện 15 lượt đánh giá mỗi cấu hình trên tập train dùng để dò tham số. Bảng hiển thị trung bình ± độ lệch chuẩn giữa các seed cho cấu hình có Macro-F1 CV cao nhất. Điểm từng fold nằm trong các file `*_cv.csv`.

In [4]:
tuned = results.dropna(subset=["cv_macro_f1_mean"])
cv_table = tuned[["dataset", "model"]].copy()
for metric, label in metrics.items():
    mean = tuned[f"cv_{metric}_mean"].map("{:.4f}".format)
    std = tuned[f"cv_{metric}_seed_std"].map("{:.4f}".format)
    cv_table[label] = mean + " ± " + std
display(cv_table)

,dataset,model,Accuracy,Precision (macro),Recall (macro),F1-Score (macro),MSE (one-hot)
0,incribo,gaussian_nb,0.3354 ± 0.0025,0.3342 ± 0.0032,0.3346 ± 0.0025,0.3096 ± 0.0022,0.4431 ± 0.0017
1,incribo,categorical_nb,0.3390 ± 0.0027,0.3389 ± 0.0029,0.3388 ± 0.0027,0.3366 ± 0.0027,0.4407 ± 0.0018
2,incribo,linear_svm,0.3391 ± 0.0009,0.3395 ± 0.0010,0.3390 ± 0.0009,0.3378 ± 0.0009,0.4406 ± 0.0006
3,incribo,rbf_svm,0.3332 ± 0.0044,0.3333 ± 0.0043,0.3332 ± 0.0044,0.3328 ± 0.0046,0.4446 ± 0.0030
7,cicids2017,gaussian_nb,0.9878 ± 0.0014,0.9701 ± 0.0006,0.7499 ± 0.0315,0.7686 ± 0.0247,0.0081 ± 0.0009
8,cicids2017,categorical_nb,0.9981 ± 0.0000,0.9934 ± 0.0002,0.9654 ± 0.0000,0.9787 ± 0.0001,0.0013 ± 0.0000
9,cicids2017,linear_svm,0.9991 ± 0.0000,0.9862 ± 0.0001,0.9945 ± 0.0002,0.9903 ± 0.0002,0.0006 ± 0.0000
10,cicids2017,rbf_svm,0.9995 ± 0.0000,0.9930 ± 0.0003,0.9933 ± 0.0006,0.9931 ± 0.0004,0.0003 ± 0.0000


## Ma trận nhầm lẫn và chỉ số từng lớp

Chọn một mô hình trong ô dưới rồi chạy lại ô đó để xem chi tiết. Hàng trong ma trận là nhãn thật, cột là nhãn dự đoán.

Các giá trị `MODEL` cho bốn mô hình là `gaussian_nb`, `categorical_nb`, `linear_svm` và `rbf_svm`.

In [10]:
MODEL = "rbf_svm"

for dataset in results["dataset"].unique():
    dataset_dir = results_path.parent / dataset
    print(dataset, MODEL)
    display(pd.read_csv(dataset_dir / f"{MODEL}_confusion.csv", index_col=0))
    display(pd.read_csv(dataset_dir / f"{MODEL}_class_metrics.csv", index_col=0).round(4))

incribo rbf_svm


,DDoS,Malware,Intrusion
actual / predicted,,,
DDoS,938,848,900
Malware,892,893,876
Intrusion,915,858,880


,precision,recall,f1-score,support
label,,,,
DDoS,0.3417,0.3492,0.3454,2686.0
Malware,0.3436,0.3356,0.3395,2661.0
Intrusion,0.3313,0.3317,0.3315,2653.0


cicids2017 rbf_svm


,DDoS,Malware,Intrusion
actual / predicted,,,
DDoS,25434,1,3
Malware,0,275,2
Intrusion,4,0,421


,precision,recall,f1-score,support
label,,,,
DDoS,0.9998,0.9998,0.9998,25438.0
Malware,0.9964,0.9928,0.9946,277.0
Intrusion,0.9883,0.9906,0.9894,425.0
